In [ ]:
Variable:
attendance = 500
print("attendance")
print("aaj ki attendence :",attendence)

In [ ]:
Calculation:
attendance = 500
demand = attendance * 0.85
production = demand * 1.05
print("Demand :", demand)
print("Production:",production)

In [ ]:
attendances = [450, 500, 520, 480]

for a in attendances:
    print("Attendance", a, "ka demand:", a * 0.85)

In [ ]:
def predict_demand(attendance):
    return attendance * 0.85

print(predict_demand(500))
print(predict_demand(300))

In [3]:
# Import the two libraries we need
import pandas as pd
import numpy as np

# Set a seed so the random data is the same each time we run the script
np.random.seed(42)

# Create 400 dates, starting from 1 January 2025
dates = pd.date_range(start="2025-01-01", periods=400, freq="D")

# Get the weekday for each date: Monday=0, Sunday=6
day_of_week = dates.dayofweek

# Randomly mark about 8% of days as holidays
is_holiday = np.random.choice([0, 1], size=400, p=[0.92, 0.08])

# Choose a menu type for each day
menu_type = np.random.choice(["regular", "special", "light"], size=400)

# Generate attendance between 300 and 600, with fewer people on Sundays and holidays
attendance = np.random.randint(300, 601, size=400)
attendance = np.where(day_of_week == 6, attendance - 100, attendance)
attendance = np.where(is_holiday == 1, attendance - 80, attendance)
attendance = np.clip(attendance, 200, 600)

# Set the attendance percentage for each menu type
menu_factor = np.select(
    [menu_type == "special", menu_type == "light"],
    [0.90, 0.80],
    default=0.85
)

# Estimate plates consumed: attendance percentage plus a little random noise
plates_consumed = np.round(
    attendance * menu_factor + np.random.normal(0, 8, size=400)
).astype(int)

# Keep plate counts within sensible limits
plates_consumed = np.clip(plates_consumed, 0, attendance)

# Create the previous day's consumption column
prev_day_consumption = np.roll(plates_consumed, 1)
prev_day_consumption[0] = 0  # No previous day is available for the first row

# Put all columns together into a table
data = pd.DataFrame({
    "date": dates,
    "day_of_week": day_of_week,
    "attendance": attendance,
    "menu_type": menu_type,
    "is_holiday": is_holiday,
    "prev_day_consumption": prev_day_consumption,
    "plates_consumed": plates_consumed
})

# Create the data folder and save the table as a CSV file
import os
os.makedirs("data", exist_ok=True)
data.to_csv("data/demand_data.csv", index=False)

# Print the first 5 rows to check the data
print(data.head())

        date  day_of_week  attendance menu_type  is_holiday  \
0 2025-01-01            2         341   regular           0   
1 2025-01-02            3         320   special           1   
2 2025-01-03            4         305   regular           0   
3 2025-01-04            5         479   regular           0   
4 2025-01-05            6         481   special           0   

   prev_day_consumption  plates_consumed  
0                     0              295  
1                   295              296  
2                   296              252  
3                   252              408  
4                   408              443  


In [4]:
data["plates_produced"] = (data["attendance"] * 0.95 + np.random.randint(-10, 30, size=len(data))).round().astype(int)
data["surplus"] = (data["plates_produced"] - data["plates_consumed"]).clip(lower=0)
print(data.head())


        date  day_of_week  attendance menu_type  is_holiday  \
0 2025-01-01            2         341   regular           0   
1 2025-01-02            3         320   special           1   
2 2025-01-03            4         305   regular           0   
3 2025-01-04            5         479   regular           0   
4 2025-01-05            6         481   special           0   

   prev_day_consumption  plates_consumed  plates_produced  surplus  
0                     0              295              326       31  
1                   295              296              315       19  
2                   296              252              298       46  
3                   252              408              478       70  
4                   408              443              460       17  


In [5]:
import os
os.makedirs("data", exist_ok=True)
data.to_csv("data/demand_data.csv", index=False)
print("Saved!")

Saved!
